In [104]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from pfns.bar_distribution import FullSupportBarDistribution
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from tabpfn_time_series import (
    FeatureTransformer,
    TabPFNMode,
    TabPFNTimeSeriesPredictor,
    TabPFNTSPipeline,
    TimeSeriesDataFrame,
)
from tabpfn_time_series.features import (
    AutoSeasonalFeature,
    CalendarFeature,
    RunningIndexFeature,
)
from tabpfn_time_series.plot import plot_forecast
from tqdm import tqdm

from tfmplayground import NanoTabPFNClassifier, NanoTabPFNRegressor
from tfmplayground.models.nanotabpfn import NanoTabPFNModel


In [105]:
DATAPATH = "/data/PFN/Mouse/"
FILE = "data_v4.csv"
VERSUCHSREIHE = [3, 7, 10, 12, 15]
NUM_SERIES = 4000
NUM_TEST_SERIES = 4000
PREDICTION_LENGTH = 3

## Prepare data

### Read data

In [106]:
df = pd.read_csv(os.path.join(DATAPATH, FILE))

# df = df[df["Versuchsreihe"].isin(VERSUCHSREIHE)].drop(columns=["Versuchsreihe"]).reset_index(drop=True)
df = df[df["Versuchsreihe"].isin(VERSUCHSREIHE)].reset_index(drop=True)

# Filter animals that have less than PREDICTION_LENGTH + 1 measurements
df = df.groupby("IdTier").filter(lambda x: len(x) > PREDICTION_LENGTH + 1).reset_index(drop=True)

# Filter animals that have values that are not possible < 5 or > 60
df = df.groupby("IdTier").filter(lambda x: (x["Gewicht"] > 5).all() and (x["Gewicht"] < 60).all()).reset_index(drop=True)

df["IdTier"] = df["IdTier"].astype("category").cat.codes.astype(int)

# Limit df to NUM_SERIES animals for now
NUM_SERIES = min(NUM_SERIES, df["IdTier"].nunique())
selected_ids = np.random.choice(df["IdTier"].unique(), size=NUM_SERIES, replace=False)
df = df[df["IdTier"].isin(selected_ids)].reset_index(drop=True)

# Ensure it is sorted
df = df.sort_values(["IdTier", "Datum"]).reset_index(drop=True)

# Rename to item_id, timestamp, target
df = df.rename(columns={"IdTier": "item_id", "Datum": "timestamp", "Gewicht": "target"})

In [107]:
# # TEMPORARY: Filter out animals with less than X measurements
# df = df.groupby("item_id").filter(lambda x: len(x) >= 20).reset_index(drop=True)

# # And have atleast one intervention
# df = df.groupby("item_id").filter(lambda x: (x["Intervention"] == 1).any()).reset_index(drop=True)

### Preprocess data

In [108]:
# Take last PREDICTION_LENGTH rows as test_df, rest as context_df
NUM_TEST_SERIES = min(NUM_TEST_SERIES, df["item_id"].nunique())
selected_ids = np.random.choice(df["item_id"].unique(), size=NUM_TEST_SERIES, replace=False)
test_df = df[df["item_id"].isin(selected_ids)].groupby("item_id").tail(PREDICTION_LENGTH)
# test_df = df.groupby("item_id").tail(PREDICTION_LENGTH)
context_df = df.drop(test_df.index)
future_df = test_df.drop(columns=["target"]).copy()

# Reset index for all DataFrames
context_df = context_df.reset_index(drop=True)
future_df = future_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Copy item_id as new column "IdTier"
context_df["IdTier"] = context_df["item_id"]
future_df["IdTier"] = future_df["item_id"]
test_df["IdTier"] = test_df["item_id"]

base_columns = ["item_id", "timestamp", "target"]
covariate_columns = ["IdTier", "Age", "Intervention"]

context_df = context_df[base_columns + covariate_columns]
future_df = future_df[base_columns[:2] + covariate_columns]
test_df = test_df[base_columns + covariate_columns]

In [109]:
# Causes of death
causes_of_death = df.groupby("item_id").nth(0).reset_index()[["item_id", "Todesursache"]]

# Therapy types:
# 0: No therapy
# 1: Radiation therapy (Bestrahlungen) only
# 2: Chemotherapy (Chemotherapie) only
# 3: Surgery (Operationen) only
# 4: Combination of therapies
# Get sum of columns "Bestrahlungen", "Chemotherapie", "Operationen" for each item_id
def determine_therapy_type(row):
    if row["Bestrahlungen"] == 0 and row["Chemotherapie"] == 0 and row["Operationen"] == 0:
        return 0
    elif row["Bestrahlungen"] > 0 and row["Chemotherapie"] == 0 and row["Operationen"] == 0:
        return 1
    elif row["Bestrahlungen"] == 0 and row["Chemotherapie"] > 0 and row["Operationen"] == 0:
        return 2
    elif row["Bestrahlungen"] == 0 and row["Chemotherapie"] == 0 and row["Operationen"] > 0:
        return 3
    else:
        return 4

therapy_types = {
    0: "No treatment",
    1: "Radiotherapy",
    2: "Chemotherapy",
    3: "Surgery",
    4: "Radio- and chemotherapy"
}

therapy_type = df.groupby("item_id")[["Bestrahlungen", "Chemotherapie", "Operationen"]].sum().reset_index()
therapy_type["type"] = therapy_type.apply(determine_therapy_type, axis=1)

# Info dataframe Versuchsreihe, causes_of_death and therapy_type
info_df = df.groupby("item_id").nth(0).reset_index()[["item_id", "Versuchsreihe"]].merge(
    causes_of_death, on="item_id"
).merge(
    therapy_type[["item_id", "type"]], on="item_id"
)

In [110]:
context_df = TimeSeriesDataFrame(context_df)
future_df["target"] = np.nan
future_df = TimeSeriesDataFrame(future_df)

In [111]:
selected_features = [
    RunningIndexFeature(),
    # CalendarFeature(),
    # AutoSeasonalFeature(),
]

feature_transformer = FeatureTransformer(selected_features)

train_tsdf, test_tsdf = feature_transformer.transform(context_df, future_df)

train_tsdf = train_tsdf.droplevel("timestamp")
test_tsdf = test_tsdf.droplevel("timestamp")

train_tsdf = train_tsdf[["IdTier", "running_index", "Intervention", "target"]]
test_tsdf = test_tsdf[["IdTier", "running_index", "Intervention", "target"]]

## Analysis

In [112]:
# Setup table showing comparison with Eagle eye
table_mae = pd.DataFrame({
    "Model": "Eagle eye (T=1)",
    "No treatment": np.array([np.nan]),
    "Radiotherapy": np.array([1.041]),
    "Chemotherapy": np.array([0.851]),
    "Surgery": np.array([np.nan]),
    "Radio- and chemotherapy": np.array([1.434]),
    "Overall": np.array([1.1087])
})

table_mape = pd.DataFrame({
    "Model": "Eagle eye (T=1)",
    "No treatment": np.array([np.nan]),
    "Radiotherapy": np.array([0.049]),
    "Chemotherapy": np.array([0.036]),
    "Surgery": np.array([np.nan]),
    "Radio- and chemotherapy": np.array([0.068]),
    "Overall": np.array([0.051])
})

### Functions

In [113]:
def get_model(
    model_path,
    model_file,
    bucket_name
):
    # Initialize a classifier
    model = NanoTabPFNModel(
        num_attention_heads=6,
        embedding_size=192,
        mlp_hidden_size=768,
        num_layers=6,
        num_outputs=100,
    )
    model.load_state_dict(
        torch.load(os.path.join(model_path, model_file))
    )
    bucket_edges = torch.load(os.path.join(model_path, bucket_name))
    dist = FullSupportBarDistribution(bucket_edges)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    reg = NanoTabPFNRegressor(
        model=model,
        dist=dist,
        device=device,
    )
    
    return reg

In [114]:
def mae(
    preds: np.ndarray,
    target: np.ndarray
):
    if not preds.shape == target.shape:
        raise ValueError(f"Preds and target must have the same shape, but got {preds.shape} and {target.shape}")
    
    if preds.ndim == 1:
        return np.abs(preds - target)
    elif preds.ndim == 2:
        return np.mean(np.abs(preds - target), axis=0)
    else:
        raise ValueError("Must be 1 or 2 dimensional")

def mape(
    preds: np.ndarray,
    target: np.ndarray
):
    if not preds.shape == target.shape:
        raise ValueError(f"Preds and target must have the same shape, but got {preds.shape} and {target.shape}")
    
    if preds.ndim == 1:
        return np.abs((preds - target) / target)
    elif preds.ndim == 2:
        return np.mean(np.abs((preds - target) / target), axis=0)
    else:
        raise ValueError("Must be 1 or 2 dimensional")

def plot_prediction(
    id_tier: int,
    data: pd.DataFrame,
    train_x: np.ndarray,
    train_y: np.ndarray,
    test_x: np.ndarray,
    test_y: np.ndarray,
    preds: np.ndarray
):
    row_selected_mouse = data[data["IdTier"] == id_tier].index[0]

    plt.plot(train_x[row_selected_mouse:, 1], train_y[row_selected_mouse:], label='context', zorder=3)
    plt.plot(test_x[..., 1], preds, color='green', label='pfn')
    for i in range(len(train_x[row_selected_mouse:])):
        if train_x[i, 2] == 1:
            plt.axvline(x=train_x[i, 1], color='red', linestyle='--', alpha=0.5, label='intervention' if i == 0 else None, zorder=1)

    for i in range(len(test_x)):
        if test_x[i, 2] == 1:
            plt.axvline(x=test_x[i, 1], color='red', linestyle='--', alpha=0.5, label='intervention' if i == 0 else None, zorder=1)

    plt.plot(test_x[..., 1], test_y, label='remaining data', zorder=2)
    plt.legend()
    plt.show()
    

In [115]:
def select_similar_mice(
    id_tier: int,
    info_df: pd.DataFrame,
    train_tsdf: TimeSeriesDataFrame,
    test_tsdf: TimeSeriesDataFrame,
    test_df: pd.DataFrame
):
    versuchsreihe_mouse = info_df[info_df["item_id"] == id_tier]["Versuchsreihe"].values[0]
    type_mouse = info_df[info_df["item_id"] == id_tier]["type"].values[0]

    similar_mice = info_df
    # Select mice from the same Versuchsreihe but not id_tier
    similar_mice = similar_mice[(similar_mice["Versuchsreihe"] == versuchsreihe_mouse) & (similar_mice["item_id"] != id_tier)]
    # Select mice with the same therapy type as the main mouse
    similar_mice = similar_mice[similar_mice["type"] == type_mouse]
    
    similar_mice = similar_mice["item_id"].unique()
    similar_mice = np.random.choice(similar_mice, size=min(30, len(similar_mice)), replace=False)

    # similar_mice dataframe contains all data for the selected mice from train_tsdf, test_tsdf and test_df
    similar_mice_train_df = train_tsdf[train_tsdf["IdTier"].isin(similar_mice)]
    similar_mice_test_tsdf = test_tsdf[test_tsdf["IdTier"].isin(similar_mice)]

    for mouse in similar_mice:
        similar_mice_test_tsdf.loc[mouse, "target"] = test_df[test_df["item_id"] == mouse]["target"].values

    combined_similar_mice_df = pd.concat([similar_mice_train_df, similar_mice_test_tsdf], ignore_index=True)
    combined_similar_mice_df = combined_similar_mice_df.sort_values(["IdTier", "running_index"]).reset_index(drop=True)
    
    return combined_similar_mice_df

def predict_mouse(
    id_tier: int,
    reg: NanoTabPFNRegressor,
    info_df: pd.DataFrame,
    train_tsdf: TimeSeriesDataFrame,
    test_tsdf: TimeSeriesDataFrame,
    test_df: pd.DataFrame,
    auto_regressive: bool = False,
    length_boundary: int | None = None
):
    temp_train_tsdf = train_tsdf[train_tsdf["IdTier"] == id_tier]
    mouse_length = len(temp_train_tsdf) + PREDICTION_LENGTH
    
    if not auto_regressive and (length_boundary is None or mouse_length >= length_boundary):
        similar_mice = select_similar_mice(
            id_tier=id_tier,
            info_df=info_df,
            train_tsdf=train_tsdf,
            test_tsdf=test_tsdf,
            test_df=test_df
        )

        temp_train_tsdf = pd.concat([similar_mice, temp_train_tsdf], ignore_index=True)

    temp_test_tsdf = test_tsdf[test_tsdf["IdTier"] == id_tier]
    temp_test_df = test_df[test_df["IdTier"] == id_tier]

    train_x = temp_train_tsdf[["IdTier", "running_index", "Intervention"]].values
    train_y = temp_train_tsdf["target"].values
    test_x = temp_test_tsdf[["IdTier", "running_index", "Intervention"]].values
    test_y = temp_test_df["target"].values

    # Reset IdTier values
    id_tier_mapping = {id_tier: idx for idx, id_tier in enumerate(temp_train_tsdf["IdTier"].unique())}
    train_x[:, 0] = np.array([id_tier_mapping[id] for id in train_x[:, 0]])
    test_x[:, 0] = np.array([id_tier_mapping[id] for id in test_x[:, 0]])

    with torch.no_grad():
        reg.fit(
            X_train=train_x,
            y_train=train_y,
        )
        preds = reg.predict(
            X_test=test_x,
        )
    
    return preds, test_y

In [116]:
def predict_all_mice(
    reg: NanoTabPFNRegressor,
    therapy_types: dict,
    info_df: pd.DataFrame,
    train_tsdf: TimeSeriesDataFrame,
    test_tsdf: TimeSeriesDataFrame,
    test_df: pd.DataFrame,
    auto_regressive: bool = False,
    length_boundary: int | None = None
):
    num_mice = len(train_tsdf["IdTier"].unique())

    all_preds = np.empty((num_mice, PREDICTION_LENGTH))
    all_targets = np.empty((num_mice, PREDICTION_LENGTH))
    mice_id_therapies = {i: [] for i in therapy_types}
    mice_idx_therapies = {i: [] for i in therapy_types}

    for i, mouse in tqdm(enumerate(train_tsdf["IdTier"].unique())):
        mouse = int(mouse)

        mouse_preds, mouse_targets = predict_mouse(
            id_tier=mouse,
            reg=reg,
            info_df=info_df,
            train_tsdf=train_tsdf,
            test_tsdf=test_tsdf,
            test_df=test_df,
            auto_regressive=auto_regressive,
            length_boundary=length_boundary
        )
        
        all_preds[i] = mouse_preds
        all_targets[i] = mouse_targets
        mice_id_therapies[int(info_df.loc[info_df["item_id"] == mouse, "type"].iloc[0])] += [mouse]
        mice_idx_therapies[int(info_df.loc[info_df["item_id"] == mouse, "type"].iloc[0])] += [i]

    mae_results = np.empty((len(therapy_types.keys()), PREDICTION_LENGTH))
    mape_results = np.empty((len(therapy_types.keys()), PREDICTION_LENGTH))

    for therapy in therapy_types:
        preds = all_preds[mice_idx_therapies[therapy], :]
        targets = all_targets[mice_idx_therapies[therapy], :]
        
        mae_results[therapy] = mae(preds, targets)
        mape_results[therapy] = mape(preds, targets)
    
    return mae_results, mape_results

In [117]:
def evaluate(
    model_name,
    reg,
    table_mae,
    table_mape,
    therapy_types,
    info_df,
    train_tsdf,
    test_tsdf,
    test_df,
    auto_regressive = False,
    length_boundary = None
):
    mae_results, mape_results = predict_all_mice(
        reg=reg,
        therapy_types=therapy_types,
        info_df=info_df,
        train_tsdf=train_tsdf,
        test_tsdf=test_tsdf,
        test_df=test_df,
        auto_regressive=auto_regressive,
        length_boundary=length_boundary
    )

    for i in range(PREDICTION_LENGTH):
        mae_row = {
            "Model": f"{model_name} (T={i + 1})",
            "No treatment": mae_results[0, i],
            "Radiotherapy": mae_results[1, i],
            "Chemotherapy": mae_results[2, i],
            "Surgery": mae_results[3, i],
            "Radio- and chemotherapy": mae_results[4, i],
            "Overall": np.mean(mae_results[[1, 2, 4], i])
        }
        
        mape_row = {
            "Model": f"{model_name} (T={i + 1})",
            "No treatment": mape_results[0, i],
            "Radiotherapy": mape_results[1, i],
            "Chemotherapy": mape_results[2, i],
            "Surgery": mape_results[3, i],
            "Radio- and chemotherapy": mape_results[4, i],
            "Overall": np.mean(mape_results[[1, 2, 4], i])
        }
        
        table_mae.loc[len(table_mae)] = mae_row
        table_mape.loc[len(table_mape)] = mape_row
    
    return table_mae, table_mape

### Basic mouse
PFN trained on only the mouse prior.

In [118]:
MODELPATH = "/data/PFN/Mouse/Experiments/Mouse1/"
MODELFILE = "pretrained_mousepfn.pth"
MODELNAME = "MousePFN"
BUCKETNAME = "buckets_mousepfn.pth"

reg = get_model(
    model_path=MODELPATH,
    model_file=MODELFILE,
    bucket_name=BUCKETNAME
)

table_mae, table_mape = evaluate(
    model_name=MODELNAME,
    reg=reg,
    table_mae=table_mae,
    table_mape=table_mape,
    therapy_types=therapy_types,
    info_df=info_df,
    train_tsdf=train_tsdf,
    test_tsdf=test_tsdf,
    test_df=test_df,
    auto_regressive=False,
    length_boundary=None
)

707it [00:32, 21.79it/s]


### TabPFN -> mouse
Pretrained on TabPFN prior v1 then pretrained on mouse prior.

In [119]:
MODELPATH = "/data/PFN/Mouse/Experiments/TabPFN_Mouse1/"
MODELFILE = "pretrained_mousepfn.pth"
MODELNAME = "TabPFN-MousePFN"
BUCKETNAME = "buckets_mousepfn.pth"

reg = get_model(
    model_path=MODELPATH,
    model_file=MODELFILE,
    bucket_name=BUCKETNAME
)

table_mae, table_mape = evaluate(
    model_name=MODELNAME,
    reg=reg,
    table_mae=table_mae,
    table_mape=table_mape,
    therapy_types=therapy_types,
    info_df=info_df,
    train_tsdf=train_tsdf,
    test_tsdf=test_tsdf,
    test_df=test_df,
    auto_regressive=False,
    length_boundary=None
)

707it [00:33, 21.40it/s]


### Mouse, no context
Make the context window only contain the mouse we're predicting for. Do not add other similar mice.

In [120]:
MODELPATH = "/data/PFN/Mouse/Experiments/Mouse1/"
MODELFILE = "pretrained_mousepfn.pth"
MODELNAME = "Auto-reg MousePFN"
BUCKETNAME = "buckets_mousepfn.pth"

reg = get_model(
    model_path=MODELPATH,
    model_file=MODELFILE,
    bucket_name=BUCKETNAME
)

table_mae, table_mape = evaluate(
    model_name=MODELNAME,
    reg=reg,
    table_mae=table_mae,
    table_mape=table_mape,
    therapy_types=therapy_types,
    info_df=info_df,
    train_tsdf=train_tsdf,
    test_tsdf=test_tsdf,
    test_df=test_df,
    auto_regressive=True,
    length_boundary=None
)

707it [00:12, 55.01it/s]


### Mouse, context for long-lived mice
Only add context when the mice has a lot of data, otherwise do it auto-regressively.

In [121]:
MODELPATH = "/data/PFN/Mouse/Experiments/Mouse1/"
MODELFILE = "pretrained_mousepfn.pth"
MODELNAME = "<20 Auto-reg MousePFN"
BUCKETNAME = "buckets_mousepfn.pth"

reg = get_model(
    model_path=MODELPATH,
    model_file=MODELFILE,
    bucket_name=BUCKETNAME
)

table_mae, table_mape = evaluate(
    model_name=MODELNAME,
    reg=reg,
    table_mae=table_mae,
    table_mape=table_mape,
    therapy_types=therapy_types,
    info_df=info_df,
    train_tsdf=train_tsdf,
    test_tsdf=test_tsdf,
    test_df=test_df,
    auto_regressive=False,
    length_boundary=20
)

707it [00:24, 28.34it/s]


### Comparison

In [122]:
table_mae.sort_values("Overall")

,Model,No treatment,Radiotherapy,Chemotherapy,Surgery,Radio- and chemotherapy,Overall
10,<20 Auto-reg MousePFN (T=1),1.111853,0.848104,0.824868,0.992021,0.657951,0.776974
7,Auto-reg MousePFN (T=1),0.778512,0.768752,0.881327,0.957091,0.825399,0.825159
1,MousePFN (T=1),1.710478,1.239133,0.758287,0.998282,0.832522,0.943314
11,<20 Auto-reg MousePFN (T=2),1.401472,1.083453,1.020267,1.480631,1.031122,1.044947
8,Auto-reg MousePFN (T=2),1.010458,1.017088,1.028458,1.066771,1.095346,1.046964
0,Eagle eye (T=1),NaN,1.041000,0.851000,NaN,1.434000,1.108700
4,TabPFN-MousePFN (T=1),1.009724,1.089047,0.957395,0.708480,1.469997,1.172146
2,MousePFN (T=2),1.977038,1.402289,1.161850,1.492529,1.389601,1.317913
9,Auto-reg MousePFN (T=3),1.208622,1.602380,1.244182,1.276583,1.533543,1.460035
5,TabPFN-MousePFN (T=2),1.280868,1.426614,1.225569,0.820612,1.846907,1.499697


In [123]:
table_mape.sort_values("Overall")

,Model,No treatment,Radiotherapy,Chemotherapy,Surgery,Radio- and chemotherapy,Overall
10,<20 Auto-reg MousePFN (T=1),0.046506,0.038744,0.034339,0.043546,0.028207,0.033764
7,Auto-reg MousePFN (T=1),0.031185,0.034022,0.036600,0.041701,0.035210,0.035277
1,MousePFN (T=1),0.067144,0.053799,0.031845,0.043775,0.038364,0.041336
8,Auto-reg MousePFN (T=2),0.041289,0.045057,0.042451,0.046958,0.047151,0.044886
11,<20 Auto-reg MousePFN (T=2),0.060444,0.049390,0.042581,0.067282,0.045233,0.045735
4,TabPFN-MousePFN (T=1),0.037648,0.046545,0.039550,0.032388,0.066405,0.050833
0,Eagle eye (T=1),NaN,0.049000,0.036000,NaN,0.068000,0.051000
2,MousePFN (T=2),0.080257,0.061567,0.050868,0.067747,0.065197,0.059210
9,Auto-reg MousePFN (T=3),0.049372,0.071532,0.052302,0.057552,0.068929,0.064255
5,TabPFN-MousePFN (T=2),0.050744,0.062352,0.053052,0.037242,0.084081,0.066495
